# Notebook 8: Sensor Robustness - Accelerometer Only

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import classification_report

In [2]:
base = r"C:\Users\M Mayavan\OneDrive\Desktop\HAR_Project\UCI HAR Dataset"
train_path = base + r"\train\Inertial Signals"
test_path = base + r"\test\Inertial Signals"
acc_signals = ['body_acc_x', 'body_acc_y', 'body_acc_z']
train_list = []
for s in acc_signals:
    data = np.loadtxt(train_path + "\\" + s + "_train.txt")
    train_list.append(data)
X_train = np.stack(train_list, axis=2)
print("Train shape:", X_train.shape)
test_list = []
for s in acc_signals:
    data = np.loadtxt(test_path + "\\" + s + "_test.txt")
    test_list.append(data)
X_test = np.stack(test_list, axis=2)
print("Test shape:", X_test.shape)
y_train = np.loadtxt(base + r"\train\y_train.txt") - 1
y_test = np.loadtxt(base + r"\test\y_test.txt") - 1

Train shape: (7352, 128, 3)
Test shape: (2947, 128, 3)


In [3]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)
mean = X_train_t.mean(dim=(1, 2), keepdim=True)
std = X_train_t.std(dim=(1, 2), keepdim=True) + 1e-8
X_train_t = (X_train_t - mean) / std
mean = X_test_t.mean(dim=(1, 2), keepdim=True)
std = X_test_t.std(dim=(1, 2), keepdim=True) + 1e-8
X_test_t = (X_test_t - mean) / std

In [4]:
train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv1d(3, 32, kernel_size=9, padding=4)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.7)
        self.fc1 = nn.Linear(2048, 64)
        self.fc2 = nn.Linear(64, 6)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.drop(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [6]:
model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
for epoch in range(50):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(epoch + 1, total_loss)

1 114.73626297712326
2 57.85875967144966
3 47.05207070708275
4 42.48573485016823
5 39.23920726776123
6 38.227946639060974
7 37.184160470962524
8 36.55852101743221
9 34.84317688643932
10 34.86496031284332
11 33.21432615816593
12 32.42028631269932
13 31.278065264225006
14 31.568359151482582
15 30.582704856991768
16 30.641049072146416
17 30.25520759820938
18 30.008455738425255
19 29.61332492530346
20 28.854038953781128
21 27.54255197942257
22 28.165146969258785
23 28.45412314683199
24 25.646014377474785
25 26.162604428827763
26 26.398766443133354
27 25.58657508343458
28 26.166571646928787
29 24.857190012931824
30 23.663880236446857
31 24.88379544019699
32 24.394464246928692
33 23.453301645815372
34 22.847371995449066
35 23.323604680597782
36 24.277865901589394
37 23.691335573792458
38 22.494784727692604
39 22.54491400718689
40 21.82361613959074
41 22.556835904717445
42 21.486588567495346
43 20.886107496917248
44 21.58699370920658
45 21.346366345882416
46 19.90509407222271
47 20.0646775141

In [7]:
model.eval()
all_preds = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch)
        preds = torch.argmax(output, dim=1)
        all_preds.append(preds)
all_preds = torch.cat(all_preds)
print("Sensor Robustness - Accelerometer Only (3 channels):")
print(classification_report(y_test_t, all_preds))
print("Compare: CNN with all 9 sensors = 93% accuracy")

Sensor Robustness - Accelerometer Only (3 channels):
              precision    recall  f1-score   support

           0       0.88      0.98      0.93       496
           1       0.99      0.84      0.91       471
           2       0.93      0.98      0.95       420
           3       0.63      0.54      0.58       491
           4       0.66      0.86      0.74       532
           5       0.89      0.73      0.81       537

    accuracy                           0.82      2947
   macro avg       0.83      0.82      0.82      2947
weighted avg       0.83      0.82      0.82      2947

Compare: CNN with all 9 sensors = 93% accuracy
